# Train Capsule R-GCN Embeddings

This notebook trains one capsule structural model per selected dataset. Run the setup cell, run exactly one dataset block, then run the remaining cells. Re-run the notebook with another dataset block to produce another model folder.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.adapters.datasets import get_dataset_adapter
from src.compute_eval import InMemoryComputeTracker
from src.optimized_grag.graph_construction import build_training_graph
from src.optimized_grag.models import capsule_fingerprint
from src.optimized_grag.models.rgcn import CapsuleRGCN, build_message_tensors, split_edges
from src.optimized_grag.partitioning import CapsuleConstructionPipeline
from src.optimized_grag.types import OptimizedGraphRAGConfig

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MESSAGE_BATCH_SIZE = 16_384  # bounds R-GCN message temporaries on 6 GB GPUs

OLLAMA_BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen3:4b-instruct"
device

## Embedding Note

The R-GCN structural model below learns from graph edges, so it does not directly need a text embedding model. The downstream projection and comparison notebooks use Ollama `nomic-embed-text`; `qwen3:4b-instruct` is the default local LLM when generation is enabled.


## Choose One Dataset

Run exactly one of the next three dataset cells before running the training cells.

Run this notebook once for the shared MetaQA knowledge graph, once for 2WikiMultiHopQA, and once for MultiHop-RAG. MetaQA 1/2/3-hop use the same graph, capsules, and unified mixed-hop query projector.

In [ ]:
ACTIVE_DATASET = {
    "label": "MetaQA shared knowledge graph (1/2/3-hop)",
    "artifact_name": "metaqa_shared_graph_train",
    "adapter_name": "metaqa",
    "adapter_kwargs": {"hop": "1-hop", "variant": "vanilla", "split": "train"},
    "dataset_root": "data/datasets/metaqa",
    "limit_documents": None,
    "limit_eval_samples": 5_000,
    "graph_from_documents": False,
    "chunk_size": 900,
    "chunk_overlap": 120,
    "max_triplets": 20_000,
    "capsule_config": {
        "max_region_nodes": 1000,
        "max_capsule_nodes": 250,
        "max_capsule_edges": 900,
        "max_hops": 2,
        "anchors_per_region": 5,
        "max_capsules_per_region": 5,
    },
}
ACTIVE_DATASET

In [ ]:
ACTIVE_DATASET = {
    "label": "2WikiMultiHopQA train sample",
    "artifact_name": "2wiki_train_sample",
    "adapter_name": "2wiki",
    "adapter_kwargs": {"split": "train"},
    "dataset_root": "data/datasets/2WikiMultiHopQA",
    "limit_documents": 1_200,
    "limit_eval_samples": 1_200,
    "graph_from_documents": True,
    "chunk_size": 850,
    "chunk_overlap": 100,
    "max_triplets": 8_000,
    "capsule_config": {
        "max_region_nodes": 700,
        "max_capsule_nodes": 180,
        "max_capsule_edges": 650,
        "max_hops": 2,
        "anchors_per_region": 4,
        "max_capsules_per_region": 4,
    },
}
ACTIVE_DATASET

In [ ]:
ACTIVE_DATASET = {
    "label": "MultiHop-RAG train sample",
    "artifact_name": "multihop_rag_train_sample",
    "adapter_name": "multihop_rag",
    "adapter_kwargs": {"split": "train"},
    "dataset_root": "data/datasets/MultiHop-RAG",
    "limit_documents": 1_500,
    "limit_eval_samples": 1_500,
    "graph_from_documents": True,
    "chunk_size": 850,
    "chunk_overlap": 100,
    "max_triplets": 8_000,
    "capsule_config": {
        "max_region_nodes": 700,
        "max_capsule_nodes": 180,
        "max_capsule_edges": 650,
        "max_hops": 2,
        "anchors_per_region": 4,
        "max_capsules_per_region": 4,
    },
}
ACTIVE_DATASET

In [ ]:
if "ACTIVE_DATASET" not in globals():
    raise RuntimeError("Run exactly one dataset selection cell first.")

EXPORT_DIR = ROOT / "artifacts" / "rgcn" / ACTIVE_DATASET["artifact_name"]
adapter_kwargs = dict(ACTIVE_DATASET.get("adapter_kwargs", {}))
if ACTIVE_DATASET.get("dataset_root"):
    adapter_kwargs["root"] = ROOT / ACTIVE_DATASET["dataset_root"]

bundle = get_dataset_adapter(
    ACTIVE_DATASET["adapter_name"],
    **adapter_kwargs,
).load(
    limit_documents=ACTIVE_DATASET.get("limit_documents"),
    limit_eval_samples=ACTIVE_DATASET.get("limit_eval_samples"),
)

graph, graph_report = build_training_graph(bundle, ACTIVE_DATASET)
if len(graph.edges) < 2:
    raise RuntimeError(f"Need at least 2 graph edges for R-GCN training; got {len(graph.edges)}.")

config = OptimizedGraphRAGConfig(**ACTIVE_DATASET.get("capsule_config", {}))
compute = InMemoryComputeTracker(log_stages=True)
capsule_result = CapsuleConstructionPipeline(config=config, instrumentation=compute).build(graph)
if not capsule_result.capsules:
    raise RuntimeError("Capsule construction produced no capsules. Increase limits or relax capsule config.")

{
    "dataset": ACTIVE_DATASET["label"],
    "export_dir": str(EXPORT_DIR),
    "graph": graph_report,
    "capsules": len(capsule_result.capsules),
}

In [ ]:
node_ids = sorted(graph.nodes.keys())
node_to_idx = {node_id: idx for idx, node_id in enumerate(node_ids)}
relation_types = sorted({edge.type for edge in graph.edges.values()})
rel_to_idx = {rel: idx for idx, rel in enumerate(relation_types)}

edges = [
    (node_to_idx[e.source_id], node_to_idx[e.target_id], rel_to_idx[e.type])
    for e in graph.edges.values()
    if e.source_id in node_to_idx and e.target_id in node_to_idx
]
if len(edges) < 2:
    raise RuntimeError(f"Need at least 2 indexed edges for training; got {len(edges)}.")

num_nodes = len(node_ids)
num_relations = len(relation_types)
num_nodes, num_relations, len(edges)

In [ ]:
train_edges, val_edges = split_edges(edges, seed=SEED)
# Validation edges are absent from message passing to prevent leakage.
train_message_edge_index, train_message_edge_type = build_message_tensors(
    train_edges, num_relations=num_relations, device=device
)
model = CapsuleRGCN(
    num_nodes, num_relations, dim=64, layers=2, message_batch_size=MESSAGE_BATCH_SIZE
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
len(train_edges), len(val_edges)

In [ ]:
def make_batch(edge_rows, negative_targets=None, batch_size=2048, shuffle=True):
    rows = torch.tensor(edge_rows, dtype=torch.long)
    tensors = [rows]
    if negative_targets is not None:
        tensors.append(torch.tensor(negative_targets, dtype=torch.long))
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle)


def batch_loss(rows, negative_targets=None, node_emb=None):
    rows = rows.to(device)
    left, right, relation = rows[:, 0], rows[:, 1], rows[:, 2]
    negatives = negative_targets.to(device) if negative_targets is not None else torch.randint(0, num_nodes, right.shape, device=device)
    node_emb = node_emb if node_emb is not None else model.encode(train_message_edge_index, train_message_edge_type)
    pos_logits = model.score_triples(node_emb, left, relation, right)
    neg_logits = model.score_triples(node_emb, left, relation, negatives)
    logits = torch.cat([pos_logits, neg_logits])
    labels = torch.cat([torch.ones_like(pos_logits), torch.zeros_like(neg_logits)])
    return F.binary_cross_entropy_with_logits(logits, labels)


validation_rng = np.random.default_rng(SEED + 1)
known_targets = {}
for source, target, relation in edges:
    known_targets.setdefault((source, relation), set()).add(target)

def fixed_negative(source, relation):
    for _ in range(100):
        candidate = int(validation_rng.integers(0, num_nodes))
        if candidate not in known_targets.get((source, relation), set()):
            return candidate
    return (source + 1) % num_nodes

val_negative_targets = [fixed_negative(source, relation) for source, _, relation in val_edges]

@torch.no_grad()
def evaluate(edge_rows, negative_targets):
    model.eval()
    losses = []
    node_emb = model.encode(train_message_edge_index, train_message_edge_type)
    for rows, negatives in make_batch(edge_rows, negative_targets, shuffle=False):
        losses.append(float(batch_loss(rows, negatives, node_emb=node_emb).detach().cpu()))
    reciprocal_ranks, hits1, hits3, hits10 = [], [], [], []
    ranking_rng = np.random.default_rng(SEED + 2)
    for source, target, relation in edge_rows[: min(1000, len(edge_rows))]:
        negatives = []
        while len(negatives) < min(99, num_nodes - 1):
            candidate = int(ranking_rng.integers(0, num_nodes))
            if candidate != target and candidate not in known_targets.get((source, relation), set()):
                negatives.append(candidate)
        candidates = torch.tensor([target, *negatives], dtype=torch.long, device=device)
        sources = torch.full_like(candidates, source)
        relations = torch.full_like(candidates, relation)
        scores = model.score_triples(node_emb, sources, relations, candidates)
        rank = 1 + int((scores[1:] >= scores[0]).sum().item())
        reciprocal_ranks.append(1.0 / rank)
        hits1.append(rank <= 1); hits3.append(rank <= 3); hits10.append(rank <= 10)
    metrics = {
        "val_mrr": float(np.mean(reciprocal_ranks)),
        "val_hits_at_1": float(np.mean(hits1)),
        "val_hits_at_3": float(np.mean(hits3)),
        "val_hits_at_10": float(np.mean(hits10)),
    }
    return (float(np.mean(losses)) if losses else 0.0), metrics


best_val = float("inf")
best_state = None
patience = 5
bad_epochs = 0
history = []

for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_rows = torch.tensor(train_edges, dtype=torch.long)
    loss = batch_loss(train_rows)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    val_loss, ranking_metrics = evaluate(val_edges, val_negative_targets)
    train_loss = float(loss.detach().cpu())
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, **ranking_metrics})
    print(f"epoch={epoch:03d} train={train_loss:.4f} val={val_loss:.4f} mrr={ranking_metrics['val_mrr']:.3f} hits10={ranking_metrics['val_hits_at_10']:.3f}")
    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("early stopping")
            break

if best_state is not None:
    model.load_state_dict(best_state)
best_val

In [ ]:
@torch.no_grad()
def pool_capsules():
    model.eval()
    node_emb = model.encode(train_message_edge_index, train_message_edge_type).detach().cpu().numpy()
    capsule_ids = []
    capsule_vectors = []
    capsule_search_texts = []
    capsule_node_ids = []
    capsule_edge_ids = []
    for capsule in capsule_result.capsules:
        indices = [node_to_idx[node_id] for node_id in capsule.node_ids if node_id in node_to_idx]
        if not indices:
            continue
        vector = node_emb[indices].mean(axis=0)
        vector = vector / max(np.linalg.norm(vector), 1e-12)
        capsule_ids.append(capsule.id)
        capsule_vectors.append(vector)
        capsule_search_texts.append(capsule.summary)
        capsule_node_ids.append(sorted(capsule.node_ids))
        capsule_edge_ids.append(sorted(capsule.edge_ids))
    if not capsule_vectors:
        raise RuntimeError("No capsule vectors were pooled.")
    return capsule_ids, np.vstack(capsule_vectors), capsule_search_texts, capsule_node_ids, capsule_edge_ids


capsule_ids, capsule_vectors, capsule_search_texts, capsule_node_ids, capsule_edge_ids = pool_capsules()
ordered_graph_edges = sorted(graph.edges.values(), key=lambda edge: edge.id)
artifact_capsule_fingerprint = capsule_fingerprint(capsule_result.capsules)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    EXPORT_DIR / "capsule_rgcn_embeddings_v4.npz",
    capsule_ids=np.array(capsule_ids),
    embeddings=capsule_vectors,
    search_texts=np.asarray(capsule_search_texts, dtype=object),
    capsule_node_ids=np.asarray(capsule_node_ids, dtype=object),
    capsule_edge_ids=np.asarray(capsule_edge_ids, dtype=object),
    graph_edge_ids=np.asarray([edge.id for edge in ordered_graph_edges], dtype=object),
    graph_edge_sources=np.asarray([edge.source_id for edge in ordered_graph_edges], dtype=object),
    graph_edge_targets=np.asarray([edge.target_id for edge in ordered_graph_edges], dtype=object),
    graph_edge_types=np.asarray([edge.type for edge in ordered_graph_edges], dtype=object),
)
torch.save(model.state_dict(), EXPORT_DIR / "capsule_rgcn.pt")
with (EXPORT_DIR / "capsule_rgcn_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "dataset": ACTIVE_DATASET,
            "bundle_metadata": bundle.metadata,
            "graph_report": graph_report,
            "capsule_metadata": capsule_result.metadata,
            "node_ids": node_ids,
            "relation_types": relation_types,
            "capsule_fingerprint": artifact_capsule_fingerprint,
            "training_schema_version": 5,
            "message_batch_size": MESSAGE_BATCH_SIZE,
            "projection_manifest": "capsule_rgcn_embeddings_v4.npz:search_texts+membership+graph_edges",
            "seed": SEED,
            "history": history,
            "best_val_loss": float(best_val),
        },
        f,
        indent=2,
    )
EXPORT_DIR